In [38]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License.

# Neo4j Import of GraphRAG Result Parquet files

This notebook imports the results of the GraphRAG indexing process into the Neo4j Graph database for further processing, analysis or visualization. 

You can also build your own GenAI applications using Neo4j and a number of RAG strategies with LangChain, LlamaIndex, Haystack, and many other frameworks.
See: https://neo4j.com/labs/genai-ecosystem

Here is what the end result looks like:

![](https://dev.assets.neo4j.com/wp-content/uploads/graphrag-neo4j-visualization.png)

## How does it work?

The notebook loads the parquet files from the `output` folder of your indexing process and loads them into Pandas dataframes.
It then uses a batching approach to send a slice of the data into Neo4j to create nodes and relationships and add relevant properties. The id-arrays on most entities are turned into relationships. 

All operations use MERGE, so they are idempotent, and you can run the script multiple times.

If you need to clean out the database, you can run the following statement

```cypher
MATCH (n)
CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 25000 ROWS;
```

In [39]:
GRAPHRAG_FOLDER = "ragtest/output/20240910-131406/artifacts"

### Depedendencies

We only need Pandas and the neo4j Python driver with the rust extension for faster network transport.

In [40]:
# %pip install --quiet pandas neo4j-rust-ext

In [41]:
import time

import pandas as pd
from neo4j import GraphDatabase

## Neo4j Installation

You can create a free instance of Neo4j [online](https://console.neo4j.io). You get a credentials file that you can use for the connection credentials. You can also get an instance in any of the cloud marketplaces.

If you want to install Neo4j locally either use [Neo4j Desktop](https://neo4j.com/download) or 
the official Docker image: `docker run -e NEO4J_AUTH=neo4j/password -p 7687:7687 -p 7474:7474 neo4j` 

In [42]:
NEO4J_URI = "neo4j+s://2cb612fd.databases.neo4j.io"  # or neo4j+s://xxxx.databases.neo4j.io
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "8xi7IayrTvIgZ4clEwhU8gS0izmB8jqzxIR6BR5XC3s"  # your password
NEO4J_DATABASE = "neo4j"

# Create a Neo4j driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

## Batched Import

The batched import function takes a Cypher insert statement (needs to use the variable `value` for the row) and a dataframe to import.
It will send by default 1k rows at a time as query parameter to the database to be inserted.

In [43]:
def batched_import(statement, df, batch_size=1000):
    """
    Import a dataframe into Neo4j using a batched approach.

    Parameters: statement is the Cypher query to execute, df is the dataframe to import, and batch_size is the number of rows to import in each batch.
    """
    total = len(df)
    start_s = time.time()
    for start in range(0, total, batch_size):
        batch = df.iloc[start : min(start + batch_size, total)]
        result = driver.execute_query(
            "UNWIND $rows AS value " + statement,
            rows=batch.to_dict("records"),
            database_=NEO4J_DATABASE,
        )
        print(result.summary.counters)
    print(f"{total} rows in {time.time() - start_s} s.")
    return total

## Indexes and Constraints

Indexes in Neo4j are only used to find the starting points for graph queries, e.g. quickly finding two nodes to connect.
Constraints exist to avoid duplicates, we create them mostly on id's of Entity types.

We use some Types as markers with two underscores before and after to distinguish them from the actual entity types.

The default relationship type here is `RELATED` but we could also infer a real relationship-type from the description or the types of the start and end-nodes.

* `__Entity__`
* `__Document__`
* `__Chunk__`
* `__Community__`
* `__Covariate__`

In [44]:
# create constraints, idempotent operation

statements = """
create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique;
create constraint document_id if not exists for (d:__Document__) require d.id is unique;
create constraint entity_id if not exists for (c:__Community__) require c.community is unique;
create constraint entity_id if not exists for (e:__Entity__) require e.id is unique;
create constraint entity_title if not exists for (e:__Entity__) require e.name is unique;
create constraint entity_title if not exists for (e:__Covariate__) require e.title is unique;
create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique;
""".split(";")

for statement in statements:
    if len((statement or "").strip()) > 0:
        print(statement)
        driver.execute_query(statement)


create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique

create constraint document_id if not exists for (d:__Document__) require d.id is unique

create constraint entity_id if not exists for (c:__Community__) require c.community is unique

create constraint entity_id if not exists for (e:__Entity__) require e.id is unique

create constraint entity_title if not exists for (e:__Entity__) require e.name is unique

create constraint entity_title if not exists for (e:__Covariate__) require e.title is unique

create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique


## Import Process

### Importing the Documents

We're loading the parquet file for the documents and create nodes with their ids and add the title property.
We don't need to store text_unit_ids as we can create the relationships and the text content is also contained in the chunks.

In [45]:
doc_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_documents.parquet", columns=["id", "title"]
)
doc_df.head(5)

,id,title
0,d228217d26ac25399f6acba637f38bab,博士班外語能力認定標準.txt
1,d893860b8aea46b1b55f3124604e892c,碩士班畢業生論文口試及離校注意事項.txt
2,4c255893e4432983355c65a9a1e7f87c,博士班修讀辦法.txt
3,f2cea4291eb3e1d0b51a6f4588dfca3b,通訊所補助研究生出席國際學術會議辦法.txt
4,5a320407efd6010098396b27124e6d83,博士班畢業生論文口試及離校注意事項.txt
5,3e83b64d5f06481abfdf35ac23c1ddbd,博士班資格考核施行細則.txt
6,a88a085fe8cd6438107e47adb3393931,通訊所甄選預備研究生辦法.txt
7,c6b8e6fd3f4eda68bb7bdf25fc6ee3e3,通訊所碩博士班學位考試採用視訊方式辦理作業辦法.txt
8,b15e16a3a5e9309c9e2a1a731da63fae,通訊所博士精進計畫.txt
9,72a4b7b869622ba121b06519dcda6c5e,碩士班修讀辦法.txt


In [46]:
# Import documents
statement = """
MERGE (d:__Document__ {id:value.id})
SET d += value {.title}
"""

batched_import(statement, doc_df)

{'_contains_updates': True, 'labels_added': 10, 'nodes_created': 10, 'properties_set': 20}
10 rows in 0.23261523246765137 s.


10

### Loading Text Units

We load the text units, create a node per id and set the text and number of tokens.
Then we connect them to the documents that we created before.

In [48]:
text_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_text_units.parquet",
    columns=["id", "text", "n_tokens", "document_ids"],
)
text_df.head(5)

,id,text,n_tokens,document_ids
0,fda34805faebe49faccdb0ee71ce36d8,國立清華大學電機資訊學院\n通訊工程研究所博士班資格考核施行細則\n八十八年九月廿九日所務會...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
1,05771538918ac3a7f80d0c0af4913500,通過 \n一Ｏ五年五月二十三日所務會議修訂通過\n一Ｏ六年九月十二日所務會議修訂通過 ...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
2,16e29e46d2f293a207556b056c4903eb,課程如下：\n通訊理論、數位訊號處理、隨機程序（包括通訊之隨機程序、網路之隨機程序）、計算機...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
3,e4978cc6cf966d79b8e743ca1874000a,訊之最佳化方法)、統計學習、機器學習理論、深度學習、網路科學、社群網路\n上述課程以本校電機...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
4,8ed281883e7033ad47b11e8496a5c7b1,目必須為三年內所修之課程；例如，100學年度所修畢之課程，最遲必須於103學年度之資格考核截...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
5,7bc7b6d4267b80937a4db9b949045381,下：\n同學以研究構想提出口試申請，由指導教授及所長共同商議決定四位以上教師（指導教授為當然...,300,[3e83b64d5f06481abfdf35ac23c1ddbd]
6,7af98234453e8d46299830f52404d485,教授提出申請，經所長同意後行之。採用視訊方式參與口試之委員至多一位。\n申請者必須獲全體委員...,200,[3e83b64d5f06481abfdf35ac23c1ddbd]
7,f597e5bcdac633983c3cd859a79d8a29,國立清華大學電機資訊學院\n通訊工程研究所博士班修讀辦法\n八十八年九月廿九日所務會議通過\...,300,[4c255893e4432983355c65a9a1e7f87c]
8,d12c0f9ab82c53fc730ce93f670af4d6,三月二日所務會議修訂通過\n九十五年九月廿八日所務會議修訂通過\n九十七年八月廿十日所務會議...,300,[4c255893e4432983355c65a9a1e7f87c]
9,2a376ddc8af3fbff1a31484220e0af7c,�資格\n凡經本校博士班研究生入學考試通過者，得進入本所博士班修讀博士學位。\n本校學士班應...,300,[4c255893e4432983355c65a9a1e7f87c]


In [49]:
statement = """
MERGE (c:__Chunk__ {id:value.id})
SET c += value {.text, .n_tokens}
WITH c, value
UNWIND value.document_ids AS document
MATCH (d:__Document__ {id:document})
MERGE (c)-[:PART_OF]->(d)
"""

batched_import(statement, text_df)

{'_contains_updates': True, 'labels_added': 98, 'relationships_created': 98, 'nodes_created': 98, 'properties_set': 294}
98 rows in 0.7101595401763916 s.


98

### Loading Nodes

For the nodes we store id, name, description, embedding (if available), human readable id.

In [50]:
entity_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_entities.parquet",
    columns=[
        "name",
        "type",
        "description",
        "human_readable_id",
        "id",
        "description_embedding",
        "text_unit_ids",
    ],
)
entity_df.head(5)

,name,type,description,human_readable_id,id,description_embedding,text_unit_ids
0,國立清華大學,ORGANIZATION,"National Tsing Hua University, a prestigious u...",0,b45241d70f0e43fca764df95b2b81f77,"[-0.05056670308113098, -0.05718901753425598, 0...","[23e917d4d6e7b349ed81d8e0c9851f71, 39185db5e88..."
1,電機資訊學院,ORGANIZATION,College of Electrical Engineering and Computer...,1,4119fd06010c494caa07f439b333f4c5,"[-0.04062497243285179, -0.03863619267940521, -...","[b26b5ceaccc68d2539ad8d1dca59e691, d90b284feda..."
2,通訊工程研究所,ORGANIZATION,The Institute of Communications Engineering at...,2,d3835bf3dda84ead99deadbeac5d0d7d,"[-0.04870221018791199, -0.03881752863526344, -...","[23e917d4d6e7b349ed81d8e0c9851f71, 39185db5e88..."
3,博士班資格考核,EVENT,PhD qualification examination process at the I...,3,077d2820ae1845bcbb1803379a3d1eae,"[0.00042648540693335235, -0.001287491060793399...",[fda34805faebe49faccdb0ee71ce36d8]
4,所務會議,EVENT,所務會議 (Institute Affairs Meeting) is an adminis...,4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,"[-0.007771966513246298, 0.019639816135168076, ...","[06ef11b3649d09903944bfc28f53368a, 61c9d59dddf..."
5,九十一年三月廿九日,EVENT,"The entity ""九十一年三月二十九日"" refers to the date of ...",5,19a7f254a5d64566ab5cc15472df02de,"[-0.027122311294078827, 0.014178574085235596, ...","[f597e5bcdac633983c3cd859a79d8a29, fda34805fae..."
6,九十二年二月十九日,EVENT,"Institute Affairs Meeting held on February 19,...",6,e7ffaee9d31d4d3c96e04f911d0a8f9e,"[0.00028185328119434416, 0.028539787977933884,...",[fda34805faebe49faccdb0ee71ce36d8]
7,九十九年十二月六日,EVENT,"The entity ""九十九年十二月六日"" refers to the Institute...",7,f7e11b0e297a44a896dc67928368f600,"[0.00014439981896430254, 0.03548393398523331, ...","[d90b284feda405e8bd93e08898b34b4c, fda34805fae..."
8,一Ｏ二年九月十八日,EVENT,Institute Affairs Meeting held on September 18...,8,1fd3fa8bb5a2408790042ab9573779ee,"[-0.01059549767524004, 0.04045439884066582, 0....",[fda34805faebe49faccdb0ee71ce36d8]
9,一Ｏ五年一月六日,EVENT,"Institute Affairs Meeting held on January 6, 2...",9,27f9fbe6ad8c4a8b9acee0d3596ed57c,"[-0.0032276504207402468, 0.030637063086032867,...",[fda34805faebe49faccdb0ee71ce36d8]


In [51]:
entity_statement = """
MERGE (e:__Entity__ {id:value.id})
SET e += value {.human_readable_id, .description, name:replace(value.name,'"','')}
WITH e, value
CALL db.create.setNodeVectorProperty(e, "description_embedding", value.description_embedding)
CALL apoc.create.addLabels(e, case when coalesce(value.type,"") = "" then [] else [apoc.text.upperCamelCase(replace(value.type,'"',''))] end) yield node
UNWIND value.text_unit_ids AS text_unit
MATCH (c:__Chunk__ {id:text_unit})
MERGE (c)-[:HAS_ENTITY]->(e)
"""

batched_import(entity_statement, entity_df)

{'_contains_updates': True, 'labels_added': 391, 'relationships_created': 826, 'nodes_created': 391, 'properties_set': 1564}
391 rows in 2.596550703048706 s.


391

### Import Relationships

For the relationships we find the source and target node by name, using the base `__Entity__` type.
After creating the `RELATED` relationships, we set the description as attribute.

In [52]:
rel_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_relationships.parquet",
    columns=[
        "source",
        "target",
        "id",
        "rank",
        "weight",
        "human_readable_id",
        "description",
        "text_unit_ids",
    ],
)
rel_df.head(5)

,source,target,id,rank,weight,human_readable_id,description,text_unit_ids
0,國立清華大學,電機資訊學院,1b893f24eb98477aad6ce49c0f26737e,19,63.0,0,The College of Electrical Engineering and Comp...,"[b26b5ceaccc68d2539ad8d1dca59e691, d90b284feda..."
1,國立清華大學,校務資訊系統,6573bc2af4f94596a3f4452a602d6fc4,22,8.0,1,The administrative information system is part ...,[d4acf6f85f7b2c7318d11f3f35a3b837]
2,國立清華大學,畢業生離校系統,0dddcca0e5df4b16bc03a51a2d2d8e16,18,8.0,2,The graduation clearance system is part of Nat...,[d4acf6f85f7b2c7318d11f3f35a3b837]
3,國立清華大學,通訊所,df40ad480a3c47299a6c8fad05349304,29,8.0,3,The communication department is part of Nation...,[d4acf6f85f7b2c7318d11f3f35a3b837]
4,國立清華大學,學術倫理聲明書,fe98fb197d294b0b837aee8d5a98dfb1,19,7.0,4,The academic ethics statement is a document re...,[d4acf6f85f7b2c7318d11f3f35a3b837]


In [53]:
rel_statement = """
    MATCH (source:__Entity__ {name:replace(value.source,'"','')})
    MATCH (target:__Entity__ {name:replace(value.target,'"','')})
    // not necessary to merge on id as there is only one relationship per pair
    MERGE (source)-[rel:RELATED {id: value.id}]->(target)
    SET rel += value {.rank, .weight, .human_readable_id, .description, .text_unit_ids}
    RETURN count(*) as createdRels
"""

batched_import(rel_statement, rel_df)

{'_contains_updates': True, 'relationships_created': 712, 'properties_set': 4272}
712 rows in 1.0367441177368164 s.


712

### Importing Communities

For communities we import their id, title, level.
We connect the `__Community__` nodes to the start and end nodes of the relationships they refer to.

Connecting them to the chunks they orignate from is optional, as the entites are already connected to the chunks.

In [54]:
community_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_communities.parquet",
    columns=["id", "level", "title", "text_unit_ids", "relationship_ids"],
)

community_df.head(5)

,id,level,title,text_unit_ids,relationship_ids
0,10,0,Community 10,"[23e917d4d6e7b349ed81d8e0c9851f71,39185db5e88b...","[1b893f24eb98477aad6ce49c0f26737e, 6573bc2af4f..."
1,7,0,Community 7,"[06ef11b3649d09903944bfc28f53368a,61c9d59dddfb...","[c0f2dc03d8df400db4997c1a0babd6ad, 0211d61aae8..."
2,9,0,Community 9,"[05771538918ac3a7f80d0c0af4913500, 05771538918...","[efb6350e65964659bc20396c0166b296, e095cc36da7..."
3,8,0,Community 8,"[05771538918ac3a7f80d0c0af4913500,16e29e46d2f2...","[0a784e00c9464bd3aeb830b908f73170, b0966a0f455..."
4,0,0,Community 0,"[389c6122eafed89b7301a581c4139680,8d5f5c7dd80c...","[a87aa935dccf49cd98b40fb5afe7ad5c, 36870a3393f..."


In [55]:
statement = """
MERGE (c:__Community__ {community:value.id})
SET c += value {.level, .title}
/*
UNWIND value.text_unit_ids as text_unit_id
MATCH (t:__Chunk__ {id:text_unit_id})
MERGE (c)-[:HAS_CHUNK]->(t)
WITH distinct c, value
*/
WITH *
UNWIND value.relationship_ids as rel_id
MATCH (start:__Entity__)-[:RELATED {id:rel_id}]->(end:__Entity__)
MERGE (start)-[:IN_COMMUNITY]->(c)
MERGE (end)-[:IN_COMMUNITY]->(c)
RETURn count(distinct c) as createdCommunities
"""

batched_import(statement, community_df)

{'_contains_updates': True, 'labels_added': 86, 'relationships_created': 1945, 'nodes_created': 86, 'properties_set': 258}
86 rows in 1.0169572830200195 s.


86

### Importing Community Reports

Fo the community reports we create nodes for each communitiy set the id, community, level, title, summary, rank, and rank_explanation and connect them to the entities they are about.
For the findings we create the findings in context of the communities.

In [56]:
community_report_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/create_final_community_reports.parquet",
    columns=[
        "id",
        "community",
        "level",
        "title",
        "summary",
        "findings",
        "rank",
        "rank_explanation",
        "full_content",
    ],
)
community_report_df.head(5)

,id,community,level,title,summary,findings,rank,rank_explanation,full_content
0,ed01fc6d-39ec-41df-b8bc-dce2e294fc69,82,3,Overseas Joint Admission Program and Hong Kong...,The community revolves around the Overseas Joi...,[{'explanation': 'The Overseas Joint Admission...,7.5,The impact severity rating is high due to the ...,# Overseas Joint Admission Program and Hong Ko...
1,8ddb2a43-4427-40f9-bedd-d370f48a553d,83,3,本所 and Associated Academic Entities,"The community revolves around 本所, an instituti...",[{'explanation': '本所 is the central entity in ...,7.5,The impact severity rating is high due to 本所's...,# 本所 and Associated Academic Entities\n\nThe c...
2,c9e35e0e-c190-4172-9f10-a05cab0a2d08,84,3,Doctoral Program at National Tsing Hua University,The community revolves around the doctoral pro...,[{'explanation': 'The doctoral program (博士班) i...,8.5,The impact severity rating is high due to the ...,# Doctoral Program at National Tsing Hua Unive...
3,dff4ba42-6336-49eb-b61f-74aafa7bd87b,85,3,Foreign Language Proficiency Certification in ...,The community revolves around the certificatio...,[{'explanation': 'The certification of foreign...,7.5,The impact severity rating is high due to the ...,# Foreign Language Proficiency Certification i...
4,9856b33b-c81b-4e15-aacc-a1c391d33003,41,2,National Tsing Hua University Graduate Student...,The community revolves around the graduate stu...,[{'explanation': 'Students are the primary foc...,7.5,The impact severity rating is high due to the ...,# National Tsing Hua University Graduate Stude...


In [57]:
# Import communities
community_statement = """
MERGE (c:__Community__ {community:value.community})
SET c += value {.level, .title, .rank, .rank_explanation, .full_content, .summary}
WITH c, value
UNWIND range(0, size(value.findings)-1) AS finding_idx
WITH c, value, finding_idx, value.findings[finding_idx] as finding
MERGE (c)-[:HAS_FINDING]->(f:Finding {id:finding_idx})
SET f += finding
"""
batched_import(community_statement, community_report_df)

{'_contains_updates': True, 'labels_added': 565, 'relationships_created': 565, 'nodes_created': 565, 'properties_set': 2211}
86 rows in 0.6400759220123291 s.


86

### Importing Covariates

Covariates are for instance claims on entities, we connect them to the chunks where they originate from.

In [58]:
cov_df = (pd.read_parquet(f"{GRAPHRAG_FOLDER}/create_final_covariates.parquet"),)
#                         columns=["id","text_unit_id"])
cov_df.head(5)
# Subject id do not match entity ids

FileNotFoundError: [Errno 2] No such file or directory: 'ragtest/output/20240910-131406/artifacts/create_final_covariates.parquet'

In [59]:
# Import covariates
cov_statement = """
MERGE (c:__Covariate__ {id:value.id})
SET c += apoc.map.clean(value, ["text_unit_id", "document_ids", "n_tokens"], [NULL, ""])
WITH c, value
MATCH (ch:__Chunk__ {id: value.text_unit_id})
MERGE (ch)-[:HAS_COVARIATE]->(c)
"""
batched_import(cov_statement, cov_df)

NameError: name 'cov_df' is not defined

### Visualize your data

You can now [Open] Neo4j on Aura, you need to log in with either SSO or your credentials.

Or open https://workspace-preview.neo4j.io and connect to your local instance, remember the URI is `neo4j://localhost` and `neo4j` as username and `password` as password.

In "Explore" you can explore by using visual graph patterns and then explore and expand further.

In "Query", you can open the left sidebar and explore by clicking on the nodes and relationships.
You can also use the co-pilot to generate Cypher queries for your, here are some examples.

#### Show a few `__Entity__` nodes and their relationships (Entity Graph)

```cypher
MATCH path = (:__Entity__)-[:RELATED]->(:__Entity__)
RETURN path LIMIT 200
```

#### Show the Chunks and the Document (Lexical Graph)

```cypher
MATCH (d:__Document__) WITH d LIMIT 1
MATCH path = (d)<-[:PART_OF]-(c:__Chunk__)
RETURN path LIMIT 100
```

####  Show a Community and it's Entities

```cypher
MATCH (c:__Community__) WITH c LIMIT 1
MATCH path = (c)<-[:IN_COMMUNITY]-()-[:RELATED]-(:__Entity__)
RETURN path LIMIT 100
```

#### Show everything

```cypher
MATCH (d:__Document__) WITH d LIMIT 1
MATCH path = (d)<-[:PART_OF]-(:__Chunk__)-[:HAS_ENTIY]->()-[:RELATED]-()-[:IN_COMMUNITY]->()
RETURN path LIMIT 250
```

We showed the visualization of this last query at the beginning.

If you have questions, feel free to reach out in the GraphRAG discord server: 
https://discord.gg/graphrag